# Preprocessing
Let's begin by initialize all of the .csv content and importing the appropriate libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/cumulative.csv')
df = df[df['koi_disposition'] != 'CANDIDATE']
df['label'] = (df['koi_disposition'] == 'CONFIRMED').astype(int)

print(f"Loaded {df.shape[0]} rows")

Loaded 7316 rows


Based on the EDA, we decided to keep a few specific features:
- `koi_period`
- `koi_depth`
- `koi_duration`
- `koi_prad`
- `koi_teq`
- `koi_impact`
- `koi_steff`
- `koi_slogg`
- `koi_srad`

Let's drop everything else and create a binary label column too.

In [2]:
# Features the model will use
features = ['koi_period', 'koi_depth', 'koi_duration', 'koi_prad',
            'koi_teq', 'koi_impact', 'koi_steff', 'koi_slogg', 'koi_srad']

X = df[features]
y = df['label']

print(f"Features shape: {X.shape}")
print(f"Label distribution:\n{y.value_counts()}")

Features shape: (7316, 9)
Label distribution:
label
0    5023
1    2293
Name: count, dtype: int64


In the EDA notebook, we took note of some missing values. We can handle them by filling them in (i.e Imputation). More info can be found [here](https://en.wikipedia.org/wiki/Imputation_(statistics)). The simplest approach we can do is something known as *median imputation* where we replace each missing value with the median of that column. There's another method known as the *mean imputation* but we will avoid using that because the dataset contains lots of outliers as shown in the box plots.

In [8]:
imputer = SimpleImputer(strategy='median')

In the EDA notebook, we also noticed how skewed the histograms were initially and how we applied a `log(1 + n)` transformation to it to squash extreme values while expanding smaller values. We will do the same thing here too. We will only apply the logarithmic transformation to features that are heavily skewed by comparing the ratio between the feature's mean and its median. Looking back on the EDA notebook, we can take note of the few features that are heavily skewed:
- `koi_period`
- `koi_depth`
- `koi_duration`
- `koi_prad`
- `koi_srad`

These are the features that we are going to apply the logarithmic transformation on

In [9]:
log_features = ['koi_period', 'koi_depth', 'koi_duration', 'koi_prad', 'koi_srad']

X_transformed = X.copy()
for col in log_features:
    X_transformed[col] = np.log1p(X_transformed[col])

When we looked at the `.describe()` table in the EDA notebook, we found that some values have extreme ranges while others are little in comparison. For example, `koi_steff` was in the thousands whereas `koi_impact` was between 0 and 1. We've got to scale the values properly to avoid situations in which the model thinks that because a feature has bigger values, it is more important.

In [10]:
scaler = StandardScaler()

Now comes the training. To test the strength of our model, we will train the model on 80% of our dataset and test it on the remaining 20%. This will help us figure out the performance of the model. 

**Sidenote:**
We can consider the model as like a mathematical function *y = f(x)*. *y* represents the probability in decimal while *x* represents all of the data inputs (i.e the training data's feature). 

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X_transformed, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Training label distribution:\n{y_train.value_counts(normalize=True)}")
print(f"Test label distribution:\n{y_test.value_counts(normalize=True)}")

Training set: 5852 samples
Test set: 1464 samples
Training label distribution:
label
0    0.686603
1    0.313397
Name: proportion, dtype: float64
Test label distribution:
label
0    0.686475
1    0.313525
Name: proportion, dtype: float64


Notice how in the training set, the percentage between confirmed (1) and false positives (0) are nearly identical? This is because of `stratify=y` which keeps the same proportions for both training and testing. This is important to keep the evaluation fair.

Now let's build the scikit-learn pipeline. We will chain the imputer and scaler together to avoid data leakage (i.e the model score is better than it actually is). A *data leakage* is giving the model information that we did not intend for it to have initially. A pipeline helps prevent this; a pipeline is just a sequence of steps that is applied to our data and packaged as a singular object.

In [12]:
preprocessing = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

### Testing

In [13]:
X_train_processed = preprocessing.fit_transform(X_train)
X_test_processed = preprocessing.transform(X_test)

print(f"Processed training shape: {X_train_processed.shape}")
print(f"Any missing values in training: {np.isnan(X_train_processed).sum()}")
print(f"Any missing values in test: {np.isnan(X_test_processed).sum()}")
print(f"Training means (should be ~0): {X_train_processed.mean(axis=0).round(2)}")
print(f"Training stds (should be ~1): {X_train_processed.std(axis=0).round(2)}")

Processed training shape: (5852, 9)
Any missing values in training: 0
Any missing values in test: 0
Training means (should be ~0): [ 0.  0. -0. -0. -0.  0.  0. -0. -0.]
Training stds (should be ~1): [1. 1. 1. 1. 1. 1. 1. 1. 1.]


The data is now ready for training!